In [1]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 52.2 MB/s eta 0:00:00


In [47]:
%%writefile app.py
import os, sqlite3, jwt, bcrypt, datetime, time, secrets, smtplib, re, streamlit as st
import plotly.graph_objects as go
from streamlit_option_menu import option_menu
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# --- 1. CONFIGURATION & NEW HYPER-GLOW HOLOGRAPHIC INDIGO THEME ---
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="dark"\nprimaryColor="#00f0ff"\nbackgroundColor="#060713"\nsecondaryBackgroundColor="#0d0e25"\ntextColor="#e2e8f0"\n')

st.set_page_config(page_title="Franchise Portal", page_icon="🔮", layout="wide", initial_sidebar_state="expanded")

# Immersive Cosmic Void & Liquid Neon Holographic Palette
COLORS = {
    "bg_main": "#060713", "bg_sidebar": "#0d0e25", "bg_card": "#13163a", "bg_card_alt": "#1a1e4f",
    "text_main": "#e2e8f0", "text_heading": "#ffffff", "text_muted": "#64748b",
    "accent": "#00f0ff", "accent_alt": "#ff007f", "accent_text": "#060713",
    "border": "#3b82f6", "border_glow": "rgba(0, 240, 255, 0.4)"
}

# Fetch Environment variables from Colab Secrets safely
JWT_SECRET = os.getenv("JWT_SECRET", "super-secret-infosys-key-2026")
SENDER_EMAIL = os.getenv("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD", "")
OTP_EXPIRY_MINUTES = 5

# Custom CSS transforming the portal into a premium, fluid dark-synth aesthetic
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;700&family=Space+Grotesk:wght@500;700&display=swap');

    html, body, .stApp {{
        background: radial-gradient(circle at 50% 0%, #11143a 0%, {COLORS['bg_main']} 70%) !important;
        font-family: 'Plus Jakarta Sans', sans-serif !important;
        color: {COLORS['text_main']} !important;
    }}
    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}
    header {{ background: transparent !important; z-index: 999999 !important; }}

    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important; display: flex !important; opacity: 1 !important;
        background: linear-gradient(135deg, {COLORS['accent']}, #0072ff) !important; border: none !important;
        border-radius: 8px !important; padding: 6px !important; margin: 8px !important;
        box-shadow: 0px 4px 15px rgba(0, 240, 255, 0.3) !important;
    }}
    .block-container {{ padding: 3rem 4rem !important; max-width: 1300px; }}

    h1, h2, h3, h4 {{ font-family: 'Space Grotesk', sans-serif !important; font-weight: 700 !important; color: {COLORS['text_heading']} !important; letter-spacing: -0.5px; }}
    label p {{ font-weight: 700 !important; font-family: 'Space Grotesk', sans-serif !important; color: {COLORS['accent']} !important; text-transform: uppercase; font-size: 11px !important; letter-spacing: 1.5px; }}

    div[data-baseweb="base-input"], div[data-baseweb="select"] > div {{ background-color: transparent !important; border: none !important; }}
    div[data-baseweb="input"], div[data-baseweb="select"] {{
        background-color: {COLORS['bg_card']} !important;
        border: 1px solid rgba(255,255,255,0.1) !important;
        border-radius: 12px !important;
        box-shadow: inset 0px 2px 4px rgba(0,0,0,0.4) !important;
        transition: all 0.2s ease-in-out !important;
    }}
    div[data-baseweb="input"]:focus-within {{ border-color: {COLORS['accent']} !important; box-shadow: 0px 0px 12px {COLORS['border_glow']} !important; }}
    input, div[data-baseweb="select"] span {{ color: {COLORS['text_main']} !important; -webkit-text-fill-color: {COLORS['text_main']} !important; font-weight: 500; }}

    /* Premium Fluid Glassmorphic Buttons */
    div[data-testid="stButton"] button {{
        background: linear-gradient(135deg, {COLORS['accent']}, #0055ff) !important; color: {COLORS['accent_text']} !important;
        border: none !important; border-radius: 12px !important;
        font-family: 'Space Grotesk', sans-serif !important; font-weight: 700 !important; font-size: 13px !important;
        text-transform: uppercase; letter-spacing: 1px;
        height: 50px !important; min-height: 50px !important; display: flex !important; align-items: center !important; justify-content: center !important;
        box-shadow: 0px 4px 20px rgba(0, 240, 255, 0.25) !important; width: 100%; transition: all 0.2s cubic-bezier(0.4, 0, 0.2, 1) !important;
        white-space: normal !important; word-wrap: break-word !important; line-height: 1.2 !important; padding: 4px 16px !important;
    }}
    div[data-testid="stButton"] button:hover {{ transform: translateY(-2px) !important; box-shadow: 0px 6px 25px rgba(0, 240, 255, 0.45) !important; }}
    div[data-testid="stButton"] button:active {{ transform: translateY(1px) !important; }}

    .alt-links button {{
        background: rgba(255, 255, 255, 0.03) !important;
        color: {COLORS['text_main']} !important;
        border: 1px solid rgba(255, 255, 255, 0.1) !important;
        box-shadow: none !important;
    }}
    .alt-links button:hover {{
        border-color: {COLORS['accent_alt']} !important;
        color: {COLORS['accent_alt']} !important;
        background: rgba(255, 0, 127, 0.05) !important;
        box-shadow: 0px 4px 15px rgba(255, 0, 127, 0.15) !important;
    }}

    section[data-testid="stSidebar"] {{ background: {COLORS['bg_sidebar']} !important; border-right: 1px solid rgba(255, 255, 255, 0.05) !important; }}

    /* Neon Bioluminescent Info Cards */
    .pn-card {{
        background: linear-gradient(145deg, {COLORS['bg_card']}, #171b47);
        border: 1px solid rgba(255, 255, 255, 0.08);
        border-radius: 16px;
        padding: 28px;
        box-shadow: 0 10px 30px rgba(0,0,0,0.3);
        transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1);
    }}
    .pn-card:hover {{
        border-color: {COLORS['accent']};
        transform: translateY(-4px);
        box-shadow: 0 15px 35px rgba(0, 240, 255, 0.15);
    }}
</style>
""", unsafe_allow_html=True)


# --- 2. DATABASE INITIALIZATION ---
def get_db():
    return sqlite3.connect("infosys_portal.db", check_same_thread=False)

def hash_txt(t):
    return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()

def check_txt(t, h):
    return bcrypt.checkpw(t.encode(), h.encode()) if h else False

with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE,
        email TEXT UNIQUE,
        password_hash TEXT,
        security_question TEXT,
        security_answer_hash TEXT,
        created_at TEXT)""")

    conn.execute("""CREATE TABLE IF NOT EXISTS login_attempts (
        email TEXT PRIMARY KEY,
        attempts INTEGER,
        last_attempt REAL)""")

# --- 3. INPUT FIELD VALIDATIONS ---
def validate_email(email):
    pattern = r"^[a-zA-Z0-9._%+-]{2,}@[a-zA-Z0-9.-]{2,}\.[a-zA-Z]{2,}$"
    return re.match(pattern, email) is not None

def validate_password(pwd):
    if len(pwd) < 8: return False
    if not re.search(r"[A-Z]", pwd): return False
    if not re.search(r"[a-z]", pwd): return False
    if not re.search(r"[0-9]", pwd): return False
    if not re.search(r"[!@#$%^&*(),.?\":{}|<>_+-]", pwd): return False
    return True

# --- 4. JWT & EMAIL SERVICES ---
def make_jwt(email, username):
    payload = {"email": email, "username": username, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)}
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None

def generate_otp():
    return f"{secrets.randbelow(900000) + 100000}"

def make_otp_token(email, otp):
    payload = {"sub": email, "otp_hash": hash_txt(otp), "type": "password_reset_otp", "iat": datetime.datetime.utcnow(), "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)}
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        if payload.get("sub") != email or payload.get("type") != "password_reset_otp":
            return False, "Security token mismatch."
        if check_txt(input_otp, payload["otp_hash"]):
            return True, "Valid"
        return False, "Invalid 6-digit OTP code."
    except jwt.ExpiredSignatureError:
        return False, f"⚠️ This OTP code expired after {OTP_EXPIRY_MINUTES} minutes. Please request a new one."
    except Exception:
        return False, "Invalid or corrupted verification token."

def send_otp_email(to_email, otp):
    if not SENDER_EMAIL or not EMAIL_PASSWORD:
        return False, "Gmail credentials missing in Colab Secrets."
    msg = MIMEMultipart('alternative')
    msg['From'] = f"Franchise Support <{SENDER_EMAIL}>"
    msg['To'] = to_email
    msg['Subject'] = "Franchise Portal - Verification Code"
    msg['Date'] = formatdate(localtime=True)
    msg['Message-ID'] = make_msgid()

    text_body = f"Your verification code for Franchise Portal is: {otp}\nThis code will expire in {OTP_EXPIRY_MINUTES} minutes."
    html_body = f"""
    <html><body><div style="max-width: 500px; margin: 0 auto; border: 1px solid rgba(0,240,255,0.3); border-radius:16px; padding: 30px; text-align: center; background:#13163a; color:#e2e8f0; box-shadow: 0 10px 30px rgba(0,0,0,0.5);">
        <h3 style="font-family:'Space Grotesk',sans-serif;color:#ffffff;font-size:22px;margin-bottom:10px;">Portal Verification</h3>
        <p style="color:#64748b;">We received a request to reset your password. Use the verification code below:</p>
        <div style="background: linear-gradient(135deg, #00f0ff, #ff007f); color:#060713; font-size: 32px; font-weight: bold; font-family:'Space Grotesk',sans-serif; letter-spacing: 6px; padding: 15px 30px; border-radius:12px; margin: 20px 0; display: inline-block;">{otp}</div>
        <p style="font-size:13px;color:#64748b;">This code expires in <b style="color:#ff007f;">{OTP_EXPIRY_MINUTES} minutes</b>.</p>
    </div></body></html>
    """
    msg.attach(MIMEText(text_body, 'plain'))
    msg.attach(MIMEText(html_body, 'html'))
    try:
        s = smtplib.SMTP('smtp.gmail.com', 587)
        s.starttls()
        s.login(SENDER_EMAIL, EMAIL_PASSWORD)
        s.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        s.quit()
        return True, "Email sent successfully!"
    except Exception as e:
        return False, f"SMTP Error: {str(e)}"

# --- 5. APP FLOW ROUTING STATE ---
if "token" not in st.session_state: st.session_state.token = None
if "page" not in st.session_state: st.session_state.page = "Login"
if "reset_email" not in st.session_state: st.session_state.reset_email = None
if "reset_mode" not in st.session_state: st.session_state.reset_mode = None
if "otp_jwt" not in st.session_state: st.session_state.otp_jwt = None

def navigate(p):
    st.session_state.page = p
    st.rerun()

def auth_header(title, sub="Intelligent Analytics Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:0.5rem 0 0.5rem;">
        <div style="font-size:48px;margin-bottom:5px;filter: drop-shadow(0px 4px 12px rgba(0,240,255,0.4));">🔮</div>
        <h1 style="font-size:2rem !important;margin:0;text-transform:uppercase;letter-spacing:-0.5px;background: linear-gradient(135deg, #fff, {COLORS['accent']}); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Franchise Portal</h1>
        <p style="color:{COLORS['text_muted']};font-size:11px;margin:6px 0 16px;font-weight:600;text-transform:uppercase;letter-spacing:1px;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:2rem;background:rgba(255,255,255,0.03);color:{COLORS['accent']};border:1px solid rgba(0,240,255,0.2);border-radius:10px;padding:8px;font-weight:700;font-family:'Space Grotesk';font-size:12px;letter-spacing:0.5px;text-transform:uppercase;">{title}</div>
    """, unsafe_allow_html=True)


# --- 6. AUTHENTICATION PAGES ---
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.2, 1])
    with mid:
        with st.container(border=True):
            # --- LOGIN PAGE ---
            if st.session_state.page == "Login":
                auth_header("Sign in to your account")
                login_id = st.text_input("Username or Email address", placeholder="you@infosys.com").strip()
                pwd = st.text_input("Password", type="password", placeholder="••••••••")
                st.markdown("<br>", unsafe_allow_html=True)

                # Primary Action Button
                if st.button("Sign In →", use_container_width=True):
                    if not login_id or not pwd:
                        st.error("❌ Both Username/Email and Password are mandatory.")
                    elif login_id == "admin@llm.com" and pwd == "Admin@123":
                        st.session_state.token = make_jwt("admin@llm.com", "System Admin")
                        navigate("Admin Dashboard")
                    else:
                        with get_db() as c:
                            r = c.execute("SELECT email, password_hash, username FROM users WHERE email=? OR username=?", (login_id.lower(), login_id)).fetchone()
                        if r and check_txt(pwd, r[1]):
                            st.session_state.token = make_jwt(r[0], r[2])
                            navigate("Dashboard")
                        else:
                            st.error("❌ Something went wrong. Please check your inputs.")

                st.markdown('<div class="alt-links">', unsafe_allow_html=True)
                col_c, col_r = st.columns(2)
                if col_c.button("Create Account", use_container_width=True): navigate("Signup")
                if col_r.button("Forgot Password", use_container_width=True): navigate("Forgot")
                st.markdown('</div>', unsafe_allow_html=True)

            # --- SIGNUP PAGE ---
            elif st.session_state.page == "Signup":
                auth_header("Create an account", "Join Franchise Portal today")
                uname = st.text_input("Full name / Username", placeholder="Jane Doe").strip()
                email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
                pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters")
                confirm_pwd = st.text_input("Confirm password", type="password", placeholder="Re-enter password")
                sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
                sa = st.text_input("Your answer", placeholder="Security answer").strip()
                st.markdown("<br>", unsafe_allow_html=True)

                if st.button("Create Account & Sign In →", use_container_width=True):
                    if not uname or not email or not pwd or not confirm_pwd or not sa:
                        st.error("⚠️ All fields are mandatory.")
                    elif not validate_email(email):
                        st.error("⚠️ Invalid email layout format.")
                    elif not validate_password(pwd):
                        st.error("⚠️ Password must contain: Min 8 chars, 1 upper, 1 lower, 1 number, and 1 symbol.")
                    elif pwd != confirm_pwd:
                        st.error("❌ Confirm password does not match.")
                    else:
                        try:
                            now_ts = datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
                            with get_db() as c:
                                c.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?, ?)", (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower()), now_ts))
                            st.session_state.token = make_jwt(email, uname)
                            st.success("✅ Account registration successful!")
                            time.sleep(1)
                            navigate("Dashboard")
                        except sqlite3.IntegrityError:
                            st.error("❌ Email layout or Username has already been claimed.")

                st.markdown('<br><div class="alt-links">', unsafe_allow_html=True)
                if st.button("← Back to Sign In", use_container_width=True): navigate("Login")
                st.markdown('</div>', unsafe_allow_html=True)

            # --- FORGOT PASSWORD PAGE ---
            elif st.session_state.page == "Forgot":
                auth_header("Reset your password", "Choose your verification method")

                if not st.session_state.reset_email:
                    target_input = st.text_input("Registered Username or Email address", placeholder="Username or Email").strip()
                    st.markdown("<br>", unsafe_allow_html=True)

                    col_sq, col_otp = st.columns(2)
                    if col_sq.button("Via Security Question", use_container_width=True):
                        if not target_input: st.error("⚠️ Enter a valid Username or Email to check.")
                        else:
                            with get_db() as c:
                                r = c.execute("SELECT email, security_question FROM users WHERE email=? OR username=?", (target_input.lower(), target_input)).fetchone()
                            if r:
                                st.session_state.reset_email = r[0]
                                st.session_state.sq_question = r[1]
                                st.session_state.reset_mode = "sq"
                                st.rerun()
                            else: st.error("❌ User not found.")

                    if col_otp.button("Via Gmail OTP", use_container_width=True):
                        if not target_input: st.error("⚠️ Enter a valid Username or Email to trace setup.")
                        else:
                            with get_db() as c:
                                r = c.execute("SELECT email FROM users WHERE email=? OR username=?", (target_input.lower(), target_input)).fetchone()
                            if r:
                                email_addr = r[0]
                                code_gen = generate_otp()
                                with st.spinner("Processing generation loop..."):
                                    ok, msg = send_otp_email(email_addr, code_gen)
                                if ok:
                                    st.session_state.reset_email = email_addr
                                    st.session_state.otp_jwt = make_otp_token(email_addr, code_gen)
                                    st.session_state.reset_mode = "otp"
                                    st.success("✅ 6-digit security code transferred to your email inbox.")
                                    time.sleep(1); st.rerun()
                                else: st.error(f"❌ {msg}")
                            else: st.error("❌ Target identification profile context missing.")
                else:
                    if st.session_state.reset_mode == "sq":
                        st.info(f"❓ **Security Question:** {st.session_state.sq_question}")
                        ans = st.text_input("Your security answer", type="password").lower().strip()
                        npw = st.text_input("New secure password", type="password")
                        confirm_npw = st.text_input("Confirm secure password", type="password")

                        st.markdown("<br>", unsafe_allow_html=True)
                        if st.button("Reset Account Password →", use_container_width=True):
                            if not ans or not npw or not confirm_npw: st.error("⚠️ All fields are mandatory.")
                            elif not validate_password(npw): st.error("⚠️ Password must conform to criteria rules.")
                            elif npw != confirm_npw: st.error("❌ Passwords do not match.")
                            else:
                                with get_db() as c:
                                    r = c.execute("SELECT security_answer_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()
                                if r and check_txt(ans, r[0]):
                                    with get_db() as c:
                                        c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                    st.success("✅ Account updated successfully.")
                                    time.sleep(1.5)
                                    st.session_state.reset_email = None; navigate("Login")
                                else: st.error("❌ Incorrect security parameter match.")

                    elif st.session_state.reset_mode == "otp":
                        st.info(f"📧 Code dispatched to: **{st.session_state.reset_email}**")
                        otp_code = st.text_input("6-digit verification sequence Code", max_chars=6, placeholder="e.g. 123456")
                        npw = st.text_input("New secure password", type="password")
                        confirm_npw = st.text_input("Confirm secure password", type="password")

                        st.markdown("<br>", unsafe_allow_html=True)
                        if st.button("Verify & Overwrite Password", use_container_width=True):
                            if not otp_code or not npw or not confirm_npw: st.error("⚠️ All input blocks are required.")
                            elif not validate_password(npw): st.error("⚠️ Password must match technical rules layout checklist.")
                            elif npw != confirm_npw: st.error("❌ Passwords match validation check broken.")
                            else:
                                is_valid, validation_msg = verify_otp_token(st.session_state.otp_jwt, otp_code, st.session_state.reset_email)
                                if is_valid:
                                    with get_db() as c:
                                        c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                    st.success("🎉 Safe identity update accomplished via secure token verification.")
                                    time.sleep(1.5)
                                    st.session_state.reset_email = None; navigate("Login")
                                else: st.error(f"❌ {validation_msg}")

                st.markdown('<br><div class="alt-links">', unsafe_allow_html=True)
                if st.button("← Cancel", use_container_width=True):
                    st.session_state.reset_email = None
                    st.session_state.reset_mode = None
                    navigate("Login")
                st.markdown('</div>', unsafe_allow_html=True)

# --- 7. SECURED APP CONTENT DASHBOARDS ---
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        navigate("Login")

    current_user_email = payload["email"]
    current_username = payload["username"]

    with st.sidebar:
        st.markdown(f"""
        <div style="padding:20px 8px;text-align:center;">
            <div style="font-size:38px;filter: drop-shadow(0px 4px 10px rgba(0,240,255,0.3));margin-bottom:8px;">🔮</div>
            <div style="font-weight:700;font-family:'Space Grotesk';font-size:16px;color:{COLORS['text_heading']};text-transform:uppercase;letter-spacing:0.5px;">Franchise Portal</div>
            <div style="font-size:10px;color:{COLORS['accent']};font-weight:600;text-transform:uppercase;letter-spacing:1.5px;margin-top:6px;">{"Admin Management" if current_user_email=="admin@llm.com" else "Intelligent Analytics"}</div>
        </div><hr style="border-color:rgba(255,255,255,0.08);margin-bottom:20px;">
        """, unsafe_allow_html=True)

        menu_opts = ["Dashboard", "Logout"] if current_user_email == "admin@llm.com" else ["Dashboard", "Analytics", "Logout"]
        menu = option_menu(None, menu_opts, icons=["house", "graph-up", "box-arrow-right"][:len(menu_opts)],
                           styles={"container": {"background-color": "transparent", "padding": "0"}, "nav-link": {"font-family": "Plus Jakarta Sans", "color": COLORS['text_main'], "border-radius": "10px", "margin": "4px 0", "padding": "10px 15px"}, "nav-link-selected": {"background": f"linear-gradient(135deg, {COLORS['accent']}, #0055ff)", "color": COLORS['accent_text'], "font-family": "Space Grotesk", "font-weight": "700"}})

        if menu == "Logout":
            st.session_state.token = None
            navigate("Login")

    if current_user_email == "admin@llm.com":
        st.markdown(f"""
        <div style="background:linear-gradient(135deg, {COLORS['bg_card']}, #1c2056);border:1px solid rgba(255,255,255,0.08);border-radius:16px;padding:26px 36px;display:flex;justify-content:space-between;align-items:center;margin-bottom:30px;box-shadow:0 10px 30px rgba(0,0,0,0.3);">
            <div><h1 style="margin:0;font-size:22px !important;text-transform:uppercase;">Control Panel</h1><div style="color:{COLORS['text_muted']};font-size:13px;font-weight:500;">Admin Identity Panel</div></div>
            <div style="background:rgba(0,240,255,0.1);padding:8px 18px;border:1px solid {COLORS['accent']};border-radius:20px;font-family:'Space Grotesk';font-weight:700;color:{COLORS['accent']};font-size:11px;letter-spacing:0.5px;">🛡️ ADMIN OPERATOR</div>
        </div>
        """, unsafe_allow_html=True)

        st.subheader("Registered Users Directory")
        with get_db() as c:
            users_list = c.execute("SELECT id, username, email, created_at FROM users").fetchall()

        if users_list:
            import pandas as pd
            df = pd.DataFrame(users_list, columns=["ID", "Username", "Email Address", "Created Timestamp"])
            st.dataframe(df, use_container_width=True)
        else:
            st.info("No signed users found inside SQLite context tables currently.")

    else:
        st.markdown(f"""
        <div style="background:linear-gradient(135deg, {COLORS['bg_card']}, #1c2056);border:1px solid rgba(255,255,255,0.08);border-radius:16px;padding:26px 36px;display:flex;justify-content:space-between;align-items:center;margin-bottom:30px;box-shadow:0 10px 30px rgba(0,0,0,0.3);">
            <div><h1 style="margin:0;font-size:22px !important;letter-spacing:-0.5px;">Welcome back, {current_username}!</h1><div style="color:{COLORS['text_muted']};font-size:13px;font-weight:500;">Logged in via {current_user_email}</div></div>
            <div style="background:rgba(255,0,127,0.1);padding:8px 18px;border:1px solid {COLORS['accent_alt']};border-radius:20px;font-family:'Space Grotesk';font-weight:700;color:{COLORS['accent_alt']};font-size:11px;letter-spacing:0.5px;">👤 STANDARD MEMBER</div>
        </div>
        """, unsafe_allow_html=True)

    # Main area view handlers for standard and admin views matching sidebar selected option
    if menu == "Dashboard":
        c1, c2, c3 = st.columns(3)
        for col, icon, lbl, val in [(c1, "📄", "Documents Indexed", "128"), (c2, "🔍", "Searches Today", "47"), (c3, "📊", "Efficiency Score", "98.4%")]:
            if lbl == "Documents Indexed": target_col = c1
            elif lbl == "Searches Today": target_col = c2
            else: target_col = c3

            target_col.markdown(f"""
            <div class="pn-card" style="text-align:center;">
                <div style="font-size:36px;margin-bottom:8px;filter:drop-shadow(0 4px 8px rgba(0,0,0,0.3));">{icon}</div>
                <div style="font-family:'Space Grotesk';font-size:28px;font-weight:700;color:{COLORS['text_heading']};">{val}</div>
                <div style="color:{COLORS['text_muted']};font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:1px;margin-top:6px;">{lbl}</div>
            </div>
            """, unsafe_allow_html=True)

    elif menu == "Analytics":
        fig = go.Figure(go.Indicator(mode="gauge+number", value=94, title={"text": "System Health Status", "font": {"color": COLORS['text_heading'], "family": "Space Grotesk", "size": 15}},
                                    gauge={"axis": {"range": [0, 100], "tickcolor": COLORS['text_main']}, "bar": {"color": COLORS['accent'], "thickness": 0.25}, "bgcolor": COLORS['bg_card_alt'], "bordercolor": "rgba(255,255,255,0.05)", "borderwidth": 1}))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)", font={"color": COLORS['text_main']}, height=280, margin=dict(t=40, b=10, l=10, r=10))
        st.plotly_chart(fig, use_container_width=True)

Overwriting app.py


In [48]:
import os, time, subprocess
from pyngrok import ngrok
from google.colab import userdata

# 1. Safely retrieve setup tokens from Google Colab Secrets Engine
try:
    os.environ['EMAIL_ADDRESS'] = userdata.get('EMAIL_ADDRESS')
    os.environ['EMAIL_PASSWORD'] = userdata.get('EMAIL_PASSWORD')
    os.environ['JWT_SECRET'] = userdata.get('JWT_SECRET') if userdata.get('JWT_SECRET') else "super-secure-backup-signing-key-2026"

    ngrok_token = userdata.get('NGROK_AUTHTOKEN')
    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
except Exception as e:
    print("⚠️ Environment Token Fault: Confirm Secrets (Key symbol) includes NGROK_AUTHTOKEN, EMAIL_PASSWORD & EMAIL_ADDRESS.")

# 2. Terminate running background Streamlit threads safely
ngrok.kill()
!pkill -f streamlit
time.sleep(1)

# 3. Boot application script context on port 8501
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    env=os.environ.copy(),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)

# 4. Initialize public pipeline tunnel
try:
    public_url = ngrok.connect(8501).public_url
    print("=" * 65)
    print(f"🚀 Combined System Live URL: {public_url}")
    print("=" * 65)
    print("⏳ Server running! Press the Stop button in Colab to shut down.")

    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Closing background services safely...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("✅ System stopped down cleanly.")

⚠️ Environment Token Fault: Confirm Secrets (Key symbol) includes NGROK_AUTHTOKEN, EMAIL_PASSWORD & EMAIL_ADDRESS.
🚀 Combined System Live URL: https://uncheck-wince-traverse.ngrok-free.dev
⏳ Server running! Press the Stop button in Colab to shut down.

🛑 Closing background services safely...
✅ System stopped down cleanly.


In [43]:
from google.colab import userdata

print(userdata.get("NGROK_AUTHTOKEN"))

3FzkjaScPPGGLVEBKz3MsavN3sT_4gnS81CUXAWvnKLUxbFt2


In [44]:
from google.colab import userdata
from pyngrok import ngrok

token = userdata.get("NGROK_AUTHTOKEN")
ngrok.set_auth_token(token)

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://uncheck-wince-traverse.ngrok-free.dev" -> "http://localhost:8501"
